# 建立影像生成應用程式（Azure OpenAI）

大型語言模型不僅限於文本生成。你還可以根據文本描述生成影像。影像作為一種模式，在醫療科技、建築、旅遊、遊戲開發、行銷等多個領域都非常實用。在本課程中，我們將使用 Microsoft Foundry 上最新的 **GPT 影像** 模型。

## 學習目標

完成本課程後，你將能夠：

- 解釋影像生成是什麼以及其應用範圍。
- 了解 `gpt-image` 模型家族及其與傳統 DALL·E 模型的差異。
- 建立影像生成應用程式及編輯影像。

## 甚麼是影像生成？

影像生成模型可根據文本提示創建影像。現代模型如 `gpt-image` 在訓練期間學習文本與影像之間的關聯，然後逐步將隨機噪聲轉換成與提示相符的影像。

兩個知名的影像模型家族是：

- **`gpt-image`（OpenAI）** — 本課程使用的當代模型。它支援從文本生成影像及影像編輯（使用遮罩進行重繪）。
- **Midjourney** — 一個流行的第三方模型，擁有自己的服務和基於 Discord 的工作流程。

> 較舊的 OpenAI 影像模型 — **DALL·E 2** 與 **DALL·E 3** — 已為過時版本。DALL·E 3 現已不再支援新部署，而像 `create_variation` 這類功能只存在於 DALL·E 2。新應用請使用 `gpt-image` 模型。

在 Microsoft Foundry 上，**`gpt-image-2`** 是最新且最強大的影像模型，也是建議的預設選擇。`gpt-image-1.5` 和 `gpt-image-1-mini` 也已普遍可用。

> **重要提示：** `gpt-image` 模型回傳生成的影像是以 **base64** 格式 (`b64_json`)，而非 URL。你的程式需將 base64 字串解碼成位元組並儲存 — 不會有可用於下載的影像 URL。


## 建立你的第一個圖像生成應用程式

那麼，建立一個圖像生成應用程式需要甚麼呢？你需要以下的函式庫：

- **python-dotenv**，強烈推薦你使用這個函式庫，將你的機密存放在 *.env* 檔案中，避免寫在程式碼裡。
- **openai**，這個函式庫是用來與 OpenAI API 互動的。
- **pillow**，用於在 Python 中處理圖像。

如果還沒做，請按照 [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) 頁面的指示，建立 Microsoft Foundry 資源和模型。選擇 **gpt-image-2** 作為模型（最新的 Azure OpenAI 圖像模型；DALL·E 已是舊版）。

1. 建立一個 *.env* 檔案，內容如下：

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    這些資訊可以在 Microsoft Foundry 入口網站你的資源的「部署」(Deployments) 區段找到。



1. 將上述庫收集在一個名為 *requirements.txt* 的檔案中，內容如下：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接著，建立虛擬環境並安裝這些庫：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 對於 Windows，請使用以下命令來建立並啟動您的虛擬環境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名為 *app.py* 的檔案中加入以下程式碼：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # 匯入 dotenv
    dotenv.load_dotenv()

    # 配置 Azure OpenAI (Microsoft Foundry) 用戶端。
    # 影像模型需要最新的 API 版本 — 請查看 Foundry 文件了解你的模型所需的版本。
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 使用影像生成 API 創建圖像
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入提示文字
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 例如 "gpt-image-2"
        )
        # 設定儲存圖像的資料夾路徑
        image_dir = os.path.join(os.curdir, 'images')

        # 如果資料夾不存在，則建立它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化圖像路徑（注意檔案類型應為 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型以 base64 (b64_json) 格式返回圖像，而非 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在預設圖像檢視器中顯示圖像
        image = Image.open(image_path)
        image.show()

    # 捕捉例外狀況
    except BadRequestError as err:
        print(err)

    ```

讓我們說明這段程式碼：

- 首先，我們匯入所需的函式庫，包括 OpenAI 函式庫、dotenv 函式庫、base64 模組，以及 Pillow 函式庫。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 接著，我們從 *.env* 檔案載入環境變數。

    ```python
    # 匯入 dotenv
    dotenv.load_dotenv()
    ```

- 然後，我們設定 Azure OpenAI (Microsoft Foundry) 用戶端。

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 接著，我們生成圖片：

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入你的提示文字
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` 模型會以 **base64** 字串的形式在 `data[0].b64_json` 中回傳圖片。我們將它解碼成位元組並寫入檔案 —— 並沒有可供下載的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後，我們開啟圖片並使用標準圖片檢視器來顯示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 有關生成圖片的更多細節

讓我們來看看 `images.generate` 的參數：

- **prompt** 是用來生成圖片的文字提示。這裡是「拿著棒棒糖的兔子騎在馬上，在長滿水仙花的霧濃草地上」。
- **size** 是生成圖片的尺寸（例如 `1024x1024`、`1536x1024`、`1024x1536`，或 `"auto"`）。
- **n** 是生成圖片的數量。這裡我們生成一張。
- **model** 是您的圖片模型部署名稱（例如 `gpt-image-2`）。

> 圖片模型不接受 `temperature` 參數 —— 那是用於文字生成的控制。若想要多樣性，可再次呼叫 API；若想減少多樣性，請使您的提示更具體。

## 生成圖片的額外功能

您已經看到如何用幾行 Python 生成圖片。`gpt-image` 模型還可以<strong>編輯</strong>現有圖片。透過提供圖片、一個選擇性的<strong>遮罩</strong>（標示要改變的區域）和提示，您可以修改圖片的一部分 —— 例如，替我們的兔子加頂帽子。

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編輯亦會以 base64 格式返回
edited_image = base64.b64decode(response.data[0].b64_json)
```

基礎圖片只包含兔子；最終圖片是在兔子頭上加了帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
